# TASK 2 — Indicator 1: Nearest-Neighbor Descriptor Consistency

**Goal**: For each of the 201 descriptors, quantify whether nearby embeddings have similar descriptor values.

## Analysis for each descriptor d:
1. Using cached kNN graphs (k=10, 50, 100):
   - Compute mean NN absolute deviation: `mean |y_i(d) − y_nn(d)|`
2. Compute random baseline:
   - Sample same number of random spectrum pairs
   - Compute mean `|y_i(d) − y_rand(d)|`
3. Compute effect ratio: `ratio = random_mean / nn_mean`
4. Compute Spearman correlation:
   - embedding similarity vs `−|Δdescriptor|`
5. Repeat for:
   - **inclusive**: all k nearest neighbors
   - **exclusive**: kNN excluding other spectra from same molecule

## Outputs:
- CSV: `results/indicators/nn_descriptor_consistency.csv`
- Console summary: Top 10 and Bottom 10 descriptors by ratio (k=50, exclusive)

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from scipy.stats import spearmanr
from collections import defaultdict
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

✓ Imports complete


## 1. Setup and Load Data

In [7]:
# Paths
data_dir = Path('../data/processed')
results_dir = Path('../results/indicators')
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Results directory: {results_dir}")

Results directory: ../results/indicators


In [8]:
# Load indicator_data from TASK 0
print("Loading indicator_data cache from TASK 0...")

import joblib

cache_path = results_dir / 'indicator_data.pkl'

try:
    indicator_data = joblib.load(cache_path)
    embeddings = indicator_data['embeddings']
    descriptors = indicator_data['descriptors']
    descriptor_names = indicator_data['descriptor_names']
    spectrum_to_molecule = indicator_data['spectrum_to_molecule']
    molecule_to_spectra = indicator_data['molecule_to_spectra']
    print(f"  ✓ Loaded from disk: {cache_path}")
except FileNotFoundError:
    print(f"  ❌ Cache not found: {cache_path}")
    print(f"  Please run TASK 0 notebook first to generate the cache.")
    raise

print(f"\n  Spectra: {embeddings.shape[0]:,}")
print(f"  Embedding dimension: {embeddings.shape[1]}")
print(f"  Descriptors: {len(descriptor_names)}")

Loading indicator_data cache from TASK 0...
  ✓ Loaded from disk: ../results/indicators/indicator_data.pkl

  Spectra: 45,185
  Embedding dimension: 1024
  Descriptors: 208


## 2. Load Pre-Computed kNN Graphs from TASK 1

In [9]:
# Load pre-computed kNN graphs from TASK 1
print("Loading pre-computed kNN graphs from TASK 1...")

k_values = [10, 50, 100]
knn_graphs_inclusive = {}
knn_graphs_exclusive = {}

for k in k_values:
    # Load inclusive
    cache_path = results_dir / f'knn_k{k}_inclusive.pkl'
    try:
        knn_graphs_inclusive[k] = joblib.load(cache_path)
        print(f"  ✓ Loaded knn_k{k}_inclusive.pkl")
    except FileNotFoundError:
        print(f"  ❌ Cache not found: {cache_path}")
        print(f"  Please run TASK 1 notebook first to generate the kNN graphs.")
        raise
    
    # Load exclusive
    cache_path = results_dir / f'knn_k{k}_exclusive.pkl'
    try:
        knn_graphs_exclusive[k] = joblib.load(cache_path)
        print(f"  ✓ Loaded knn_k{k}_exclusive.pkl")
    except FileNotFoundError:
        print(f"  ❌ Cache not found: {cache_path}")
        print(f"  Please run TASK 1 notebook first to generate the kNN graphs.")
        raise

print(f"\n✓ kNN graphs loaded for k={k_values}")

Loading pre-computed kNN graphs from TASK 1...
  ✓ Loaded knn_k10_inclusive.pkl
  ✓ Loaded knn_k10_exclusive.pkl
  ✓ Loaded knn_k50_inclusive.pkl
  ✓ Loaded knn_k50_exclusive.pkl
  ✓ Loaded knn_k100_inclusive.pkl
  ✓ Loaded knn_k100_exclusive.pkl

✓ kNN graphs loaded for k=[10, 50, 100]


## 3. Analysis: NN Descriptor Consistency

In [10]:
# For efficiency, sample random pairs (don't compute all pairs)
def compute_random_baseline(descriptors, n_pairs):
    """
    Sample n_pairs random spectrum pairs and compute mean absolute deviation.
    """
    n_spectra = descriptors.shape[0]
    idx1 = np.random.choice(n_spectra, size=n_pairs, replace=True)
    idx2 = np.random.choice(n_spectra, size=n_pairs, replace=True)
    
    # Compute absolute differences
    diff = np.abs(descriptors[idx1] - descriptors[idx2])
    return diff.mean(axis=0)  # per-descriptor mean

def compute_nn_consistency(descriptors, knn_indices, knn_distances, spectrum_to_molecule=None, exclusive=False):
    """
    Compute NN descriptor consistency metrics.
    
    Args:
        descriptors: (n_spectra, n_descriptors)
        knn_indices: (n_spectra, k) - indices of k nearest neighbors
        knn_distances: (n_spectra, k) - cosine distances
        spectrum_to_molecule: array mapping spectrum idx to molecule InChIKey
        exclusive: if True, exclude same-molecule neighbors
    
    Returns:
        tuple: (mean_diff, spearman_corr) per descriptor
    """
    n_spectra, n_descriptors = descriptors.shape
    k = knn_indices.shape[1]
    
    mean_diffs = np.zeros(n_descriptors)
    spearman_corrs = np.zeros(n_descriptors)
    
    for i in range(n_spectra):
        neighbors = knn_indices[i]
        distances = knn_distances[i]
        
        # Filter by same molecule if exclusive
        if exclusive and spectrum_to_molecule is not None:
            same_mol = spectrum_to_molecule[neighbors] == spectrum_to_molecule[i]
            neighbors = neighbors[~same_mol]
            distances = distances[~same_mol]
        
        if len(neighbors) > 0:
            # Compute per-descriptor differences
            desc_diff = np.abs(descriptors[i] - descriptors[neighbors])  # (n_neighbors, n_descriptors)
            mean_diffs += desc_diff.mean(axis=0)
            
            # Spearman correlation: embedding similarity vs descriptor difference
            # Use negative distance as similarity (higher = more similar)
            similarity = 1 - distances  # convert distance to similarity
            
            for d in range(n_descriptors):
                corr, _ = spearmanr(similarity, desc_diff[:, d])
                spearman_corrs[d] += corr if not np.isnan(corr) else 0
    
    mean_diffs /= n_spectra
    spearman_corrs /= n_spectra
    
    return mean_diffs, spearman_corrs

print("✓ Analysis functions defined")

✓ Analysis functions defined


In [ ]:
# Run analysis for each k and each descriptor
print("\nRunning consistency analysis...")
print("="*80)

results = []

for k in k_values:
    print(f"\nk = {k}")
    
    # Number of pairs to sample for random baseline
    n_edges = embeddings.shape[0] * k
    random_baseline = compute_random_baseline(descriptors, n_pairs=n_edges)
    
    for graph_type in ['inclusive', 'exclusive']:
        print(f"  Computing {graph_type} neighbors...")
        
        # Get kNN graph
        if graph_type == 'inclusive':
            knn_data = knn_graphs_inclusive[k]
        else:
            knn_data = knn_graphs_exclusive[k]
        
        knn_indices = knn_data['indices']
        knn_distances = knn_data['distances']
        
        # Compute consistency metrics (no need for inclusive/exclusive flag since graphs are pre-filtered)
        n_spectra, n_descriptors = descriptors.shape
        n_neighbors = knn_indices.shape[1]
        
        mean_diffs = np.zeros(n_descriptors)
        spearman_corrs = np.zeros(n_descriptors)
        
        for i in range(n_spectra):
            neighbors = knn_indices[i].astype(int)
            distances = knn_distances[i]
            
            # Filter out invalid indices (padding from exclusive version)
            valid_mask = neighbors >= 0
            neighbors = neighbors[valid_mask]
            distances = distances[valid_mask]
            
            if len(neighbors) > 0:
                # Compute per-descriptor differences
                desc_diff = np.abs(descriptors[i] - descriptors[neighbors])
                mean_diffs += desc_diff.mean(axis=0)
                
                # Spearman correlation: similarity vs -|Δdescriptor|
                # Negate difference so positive corr = good (similar embeddings → similar descriptors)
                similarity = 1 - distances
                for d in range(n_descriptors):
                    corr, _ = spearmanr(similarity, -desc_diff[:, d])
                    spearman_corrs[d] += corr if not np.isnan(corr) else 0
        
        mean_diffs /= n_spectra
        spearman_corrs /= n_spectra
        
        # Compute effect ratio
        ratio = random_baseline / np.maximum(mean_diffs, 1e-10)
        
        # Store results
        for d, desc_name in enumerate(descriptor_names):
            results.append({
                'descriptor': desc_name,
                'k': k,
                'neighbors': graph_type,
                'nn_mean_diff': mean_diffs[d],
                'random_mean_diff': random_baseline[d],
                'effect_ratio': ratio[d],
                'spearman_corr': spearman_corrs[d]
            })

print(f"\n{'='*80}")
print(f"✓ Analysis complete")


Running consistency analysis...

k = 10
  Computing inclusive neighbors...
  Computing exclusive neighbors...

k = 50
  Computing inclusive neighbors...
  Computing exclusive neighbors...

k = 100
  Computing inclusive neighbors...
  Computing exclusive neighbors...

✓ Analysis complete


## 4. Results Aggregation

In [12]:
# Convert to dataframe
df_results = pd.DataFrame(results)

print(f"Results shape: {df_results.shape}")
print(f"\nColumns: {df_results.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_results.head(10))

Results shape: (1248, 7)

Columns: ['descriptor', 'k', 'neighbors', 'nn_mean_diff', 'random_mean_diff', 'effect_ratio', 'spearman_corr']

First few rows:
            descriptor   k  neighbors  nn_mean_diff  random_mean_diff  \
0       MaxEStateIndex  10  inclusive      0.937212          2.711796   
1       MinEStateIndex  10  inclusive      0.517522          1.268156   
2    MaxAbsEStateIndex  10  inclusive      0.937212          2.711796   
3    MinAbsEStateIndex  10  inclusive      0.095543          0.232767   
4                  qed  10  inclusive      0.087065          0.232067   
5                MolWt  10  inclusive     41.763596        161.020024   
6       HeavyAtomMolWt  10  inclusive     39.160091        149.509783   
7           ExactMolWt  10  inclusive     41.717710        160.871858   
8  NumValenceElectrons  10  inclusive     16.184601         62.947412   
9  NumRadicalElectrons  10  inclusive      0.000000          0.000000   

   effect_ratio  spearman_corr  
0      2.

In [13]:
# Save to CSV
csv_path = results_dir / 'nn_descriptor_consistency.csv'
df_results.to_csv(csv_path, index=False)
print(f"✅ Results saved to: {csv_path}")

✅ Results saved to: ../results/indicators/nn_descriptor_consistency.csv


## 5. Summary: Top and Bottom Descriptors

In [ ]:
# Filter for k=50, exclusive neighbors
df_k50_excl = df_results[(df_results['k'] == 50) & (df_results['neighbors'] == 'exclusive')].copy()
df_k50_excl = df_k50_excl.sort_values('effect_ratio', ascending=False)

print("="*80)
print("SUMMARY: Nearest-Neighbor Descriptor Consistency (k=50, exclusive)")
print("="*80)
print(f"\nTotal descriptors analyzed: {len(df_k50_excl)}")
print(f"Mean effect ratio: {df_k50_excl['effect_ratio'].mean():.3f}")
print(f"Median effect ratio: {df_k50_excl['effect_ratio'].median():.3f}")

print(f"\n{'TOP 10 — Highest Effect Ratio (NN >> random)':^80}")
print(f"{'Descriptor':<30} {'Effect Ratio':>15} {'Spearman Corr':>15}")
print("-" * 80)
for _, row in df_k50_excl.head(10).iterrows():
    print(f"{row['descriptor']:<30} {row['effect_ratio']:>15.4f} {row['spearman_corr']:>15.4f}")

print(f"\n{'BOTTOM 10 — Lowest Effect Ratio (NN << random)':^80}")
print(f"{'Descriptor':<30} {'Effect Ratio':>15} {'Spearman Corr':>15}")
print("-" * 80)
for _, row in df_k50_excl.tail(10).iterrows():
    print(f"{row['descriptor']:<30} {row['effect_ratio']:>15.4f} {row['spearman_corr']:>15.4f}")

print(f"\n{'='*80}")
print("Interpretation:")
print("  - High ratio (>>1): Descriptor is CONSISTENT in embedding neighborhood")
print("  - Low ratio (<<1): Descriptor is INCONSISTENT in embedding neighborhood")
print("  - Spearman corr: Positive = well-aligned (similar embeddings → similar descriptors)")
print("  - Spearman corr: Negative = poorly aligned (geometry doesn't capture descriptor)")
print(f"{'='*80}")

SUMMARY: Nearest-Neighbor Descriptor Consistency (k=50, exclusive)

Total descriptors analyzed: 208
Mean effect ratio: 1.406
Median effect ratio: 1.415

                  TOP 10 — Highest Effect Ratio (NN >> random)                  
Descriptor                        Effect Ratio   Spearman Corr
--------------------------------------------------------------------------------
Chi2n                                   2.1297         -0.0867
NumValenceElectrons                     2.1277         -0.0941
Chi1n                                   2.1262         -0.0912
HeavyAtomCount                          2.1216         -0.0942
Chi3n                                   2.1204         -0.0802
Chi0n                                   2.1195         -0.0916
LabuteASA                               2.1181         -0.0959
ExactMolWt                              2.1127         -0.0987
MolWt                                   2.1125         -0.0987
Chi0                                    2.1092         

## 6. Additional Analysis: Across k values

In [15]:
# Pivot to see how effect ratio varies with k
df_pivot = df_results[df_results['neighbors'] == 'exclusive'].pivot_table(
    index='descriptor',
    columns='k',
    values='effect_ratio',
    aggfunc='first'
)

# Find descriptors with highest and lowest ratios
df_pivot['mean_ratio'] = df_pivot.mean(axis=1)
df_pivot = df_pivot.sort_values('mean_ratio', ascending=False)

print(f"\n{'Effect Ratio across k values (exclusive) — Top 10 descriptors':^80}")
print(df_pivot.head(10).round(3))

print(f"\n{'Effect Ratio across k values (exclusive) — Bottom 10 descriptors':^80}")
print(df_pivot.tail(10).round(3))


         Effect Ratio across k values (exclusive) — Top 10 descriptors          
k                       10     50    100  mean_ratio
descriptor                                          
Chi2n                2.859  2.130  1.969       2.319
NumValenceElectrons  2.863  2.128  1.961       2.317
Chi3n                2.865  2.120  1.961       2.315
Chi1n                2.858  2.126  1.960       2.315
HeavyAtomCount       2.852  2.122  1.956       2.310
Chi0n                2.847  2.119  1.955       2.307
LabuteASA            2.849  2.118  1.951       2.306
ExactMolWt           2.841  2.113  1.947       2.300
MolWt                2.841  2.112  1.947       2.300
Chi0                 2.829  2.109  1.946       2.295

        Effect Ratio across k values (exclusive) — Bottom 10 descriptors        
k                       10     50    100  mean_ratio
descriptor                                          
fr_isothiocyan       1.145  0.875  0.830       0.950
fr_term_acetylene    1.002  0.839  0.964 

## 7. Task Complete

In [16]:
print("\n" + "="*80)
print("TASK 2 — Indicator 1 COMPLETE")
print("="*80)
print(f"\nResults saved:")
print(f"  - CSV: results/indicators/nn_descriptor_consistency.csv")
print(f"\nNext steps:")
print(f"  - TASK 3: Indicator 2 (Clustering Purity)")
print(f"  - TASK 4: Indicator 3 (Structural Separation)")
print(f"  - TASK 5: Generate figures and integrated analysis")
print("="*80)


TASK 2 — Indicator 1 COMPLETE

Results saved:
  - CSV: results/indicators/nn_descriptor_consistency.csv

Next steps:
  - TASK 3: Indicator 2 (Clustering Purity)
  - TASK 4: Indicator 3 (Structural Separation)
  - TASK 5: Generate figures and integrated analysis
